# Manual Price Push

Push manual price overrides for specific SKUs using market data or fixed prices.

## How to use
1. Run cells 1-3 (setup + data loading)
2. Edit the `PUSH_LIST` in cell 4 with your product IDs and desired actions
3. Run cell 4 — prices are computed per product × cohort (using cohort-specific market data)
4. Review the output table
5. Set `MODE = 'live'` and run cell 5 to push

> Each product is automatically expanded to all 9 cohorts. Market-based actions use cohort-specific prices.  
> Main/general cohorts (695, 61, 699, 697, 698, 696) are auto-mirrored by the push handler.

## Available Actions
| Action | Description |
|--------|-------------|
| `market_min` | Lowest market price (P0 / minimum) |
| `market_25` | 25th percentile market price |
| `market_50` | Median market price (P50) |
| `market_75` | 75th percentile market price |
| `market_max` | Highest market price (maximum) |
| `market_avg` | Average of min and max market prices |
| `target_margin` | Price from brand-category target margin |
| `step_up` | Next tier above current price in the price list |
| `step_down` | Next tier below current price in the price list |
| `<number>` | Fixed price (e.g. `115`, `23.5`) |

> Only SKUs with stock > 0 are pushed.

In [1]:
# =============================================================================
# SETUP
# =============================================================================
import sys, os
import pandas as pd
import numpy as np
import pytz
from datetime import datetime

sys.path.insert(0, os.path.abspath('.'))
import setup_environment_2
setup_environment_2.initialize_env()

from db import query_snowflake
from constants import WAREHOUSE_MAPPING, COHORT_IDS

os.chdir(os.path.join(os.path.dirname(os.path.abspath('__file__')) or os.getcwd(), 'modules'))
%run push_prices_handler.ipynb
os.chdir('..')

CAIRO_TZ = pytz.timezone('Africa/Cairo')
TODAY = datetime.now(CAIRO_TZ).strftime('%Y-%m-%d')
print(f'Ready. Today: {TODAY}')

/home/ec2-user/.Renviron
/home/ec2-user/service_account_key.json


/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/snowflake/connector/options.py:104: UserWarning: You have an incompatible version of 'pyarrow' installed (20.0.0), please install a version that adheres to: 'pyarrow<19.0.0; extra == "pandas"'
  warn_incompatible_dep(


Push Prices Handler loaded at 2026-04-08 11:17:45 Cairo time
✓ API credentials loaded successfully
✓ Google Sheets client initialized
Ready. Today: 2026-04-08


In [2]:
# =============================================================================
# LOAD MARKET DATA + WAC + CURRENT PRICES + PACKING UNITS
# =============================================================================
os.chdir('modules')
%run market_data_module_2.ipynb
os.chdir('..')

print('Loading market data (V2)...')
market_data = get_market_data_v2()
print(f'  Market data: {len(market_data)} rows')

print('Loading WAC...')
WAC_QUERY = f'''
SELECT product_id, wac_p
FROM finance.all_cogs c
WHERE wac_p > 0
    AND CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP()) 
        BETWEEN c.from_date AND c.to_date
'''
df_wac = query_snowflake(WAC_QUERY)
print(f'  WAC: {len(df_wac)} rows')

print('Loading current prices...')
PRICES_QUERY = f'''

    SELECT 
        cohort_id,
        pu.product_id,
        pu.packing_unit_id,
        pu.basic_unit_count,
        AVG(cpu.price) AS current_price
    FROM cohort_product_packing_units cpu
    JOIN packing_unit_products pu ON pu.id = cpu.product_packing_unit_id
    WHERE cpu.cohort_id IN ({','.join(str(c) for c in COHORT_IDS)})
        AND cpu.created_at::DATE <> '2023-07-31'
        AND cpu.is_customized = TRUE
    GROUP BY ALL

'''
df_prices = query_snowflake(PRICES_QUERY)
print(f'  Prices: {len(df_prices)} rows')

print('Loading packing units...')
PU_QUERY = '''
WITH sales_check AS (
    SELECT DISTINCT
        pso.product_id,
        pso.packing_unit_id,
        SUM(pso.total_price) AS nmv
    FROM product_sales_order pso
    JOIN sales_orders so ON so.id = pso.sales_order_id
    WHERE so.created_at >= CURRENT_DATE - 60 
        AND so.sales_order_status_id NOT IN (7, 12)
        AND so.channel IN ('telesales', 'retailer')
        AND pso.purchased_item_count <> 0
    GROUP BY 1, 2
)
SELECT product_id, packing_unit_id, basic_unit_count
FROM (
    SELECT *, MAX(nmv) OVER (PARTITION BY product_id, is_basic_unit) AS top_nmv
    FROM (
        SELECT 
            pup.product_id,
            pup.packing_unit_id,
            pup.basic_unit_count,
            pup.is_basic_unit,
            COUNT(DISTINCT CASE WHEN pup.basic_unit_count = 1 THEN pup.packing_unit_id END) 
                OVER (PARTITION BY pup.product_id) AS total_basic,
            COALESCE(sc.nmv, 0) AS nmv
        FROM packing_unit_products pup
        LEFT JOIN sales_check sc 
            ON sc.product_id = pup.product_id 
            AND sc.packing_unit_id = pup.packing_unit_id
        WHERE pup.deleted_at IS NULL
    )
)
WHERE nmv = top_nmv OR (top_nmv = 0 AND total_basic <= 1)
'''
df_pus = query_snowflake(PU_QUERY)
print(f'  Packing units: {len(df_pus)} rows')

print('Loading product info...')
PRODUCT_QUERY = '''
SELECT p.id AS product_id,
       CONCAT(p.name_ar, ' ', p.size, ' ', product_units.name_ar) AS sku,
       b.name_ar AS brand,
       c.name_ar AS cat
FROM products p
JOIN brands b ON b.id = p.brand_id
JOIN categories c ON c.id = p.category_id
JOIN product_units ON product_units.id = p.unit_id
'''
df_products = query_snowflake(PRODUCT_QUERY)
print(f'  Products: {len(df_products)} rows')

print('Loading target margins...')
MARGIN_QUERY = f'''
SELECT brand, cat, target_margin
FROM (
    SELECT b.name_ar AS brand, c.name_ar AS cat, cplan.margin AS target_margin,
           ROW_NUMBER() OVER (PARTITION BY b.name_ar, c.name_ar ORDER BY cplan.date DESC) AS rn
    FROM performance.commercial_targets cplan
    JOIN brands b ON b.id = cplan.brand_id
    JOIN categories c ON c.id = cplan.category_id
    WHERE cplan.margin IS NOT NULL
    QUALIFY CASE 
        WHEN DATE_TRUNC('month', MAX(date) OVER()) = DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE) 
        THEN DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE)
        ELSE DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - INTERVAL '1 month') 
    END = DATE_TRUNC('month', date)
)
WHERE rn = 1
'''
df_targets = query_snowflake(MARGIN_QUERY)
print(f'  Target margins: {len(df_targets)} rows')

print('Loading stocks...')
STOCKS_QUERY = '''
WITH parent_whs AS (
    SELECT * FROM (VALUES (236,343),(1,467),(962,343)) x(parent_id,child_id)
)
SELECT warehouse_id, product_id,
    CASE WHEN stocks_child IS NOT NULL AND stocks = 0 THEN stocks_child ELSE stocks END AS stocks
FROM (
    SELECT pw.warehouse_id, pw.product_id,
        pw.available_stock::INTEGER AS stocks,
        pw2.available_stock::INTEGER AS stocks_child
    FROM product_warehouse pw
    LEFT JOIN parent_whs p ON p.parent_id = pw.warehouse_id
    LEFT JOIN product_warehouse pw2 ON pw2.warehouse_id = p.child_id AND pw.product_id = pw2.product_id
    WHERE pw.warehouse_id NOT IN (6, 9, 10)
        AND pw.is_basic_unit = 1
)
'''
df_stocks = query_snowflake(STOCKS_QUERY)
print(f'  Stocks: {len(df_stocks)} rows')

print('\nAll data loaded.')

/home/ec2-user/.Renviron
/home/ec2-user/service_account_key.json
Market Data Module loaded at 2026-04-08 11:17:47 Cairo time
Snowflake timezone: America/Los_Angeles
All queries defined ✓
Helper functions defined ✓
get_market_data() function defined ✓
get_margin_tiers() function defined ✓
get_brand_market_percentiles() function defined ✓
fill_brand_market_fallback() function defined ✓

MARKET DATA MODULE READY

Available functions (NO INPUT REQUIRED):
  - get_market_data()              : Fetch and process all market prices
  - get_margin_tiers()             : Fetch and calculate margin tiers
  - get_brand_market_percentiles() : Fetch brand-level market margin percentiles
  - fill_brand_market_fallback()   : Fill NaN market cols with brand percentiles

Usage:
  %run market_data_module.ipynb
  df_market = get_market_data()
  df_tiers = get_margin_tiers()
  df_brand_percs = get_brand_market_percentiles()
  df = fill_brand_market_fallback(df, df_brand_percs)
get_market_signals() function de

In [3]:
# =============================================================================
# BUILD LOOKUP TABLE (product_id → market prices + WAC + target margin)
# =============================================================================

# V2 market_data returns (product_id, region, price_tiers, wac_p, target_margin, num_sources, market_data_source)
# Expand to cohorts for lookup
df_market_cohorts = expand_to_cohorts(market_data)

# Lookup is per (product_id, cohort_id)
lookup = df_products.merge(df_wac, on='product_id', how='left')

cohort_df = pd.DataFrame({'cohort_id': COHORT_IDS})
lookup = lookup.merge(cohort_df, how='cross')
lookup = lookup.merge(
    df_market_cohorts[['product_id', 'cohort_id', 'price_tiers', 'target_margin']],
    on=['product_id', 'cohort_id'], how='left'
)
lookup['price_tiers'] = lookup['price_tiers'].apply(lambda x: x if isinstance(x, list) else [])

# Fill target_margin for rows without V2 data
lookup = lookup.merge(df_targets, on=['brand', 'cat'], how='left', suffixes=('', '_tgt'))
for col in ['target_bm', 'target_bm_tgt', 'cat_target_margin', 'cat_target_margin_tgt']:
    if col in lookup.columns:
        lookup['target_margin'] = lookup['target_margin'].fillna(lookup[col])
lookup['target_margin'] = lookup['target_margin'].fillna(0.10)
lookup.drop(columns=[c for c in lookup.columns if c.startswith('target_bm') or c.startswith('cat_target')], inplace=True, errors='ignore')

# Derive market percentile prices from price_tiers list
def tiers_to_percentiles(tiers):
    if not tiers or len(tiers) == 0:
        return pd.Series({'market_min': np.nan, 'market_25': np.nan, 'market_50': np.nan,
                          'market_75': np.nan, 'market_max': np.nan, 'market_avg': np.nan})
    arr = np.array(tiers)
    return pd.Series({
        'market_min': arr[0],
        'market_25': np.percentile(arr, 25),
        'market_50': np.percentile(arr, 50),
        'market_75': np.percentile(arr, 75),
        'market_max': arr[-1],
        'market_avg': (arr[0] + arr[-1]) / 2,
    })

lookup = pd.concat([lookup, lookup['price_tiers'].apply(tiers_to_percentiles)], axis=1)

# Merge current base-unit price per (product_id, cohort_id)
base_prices = df_prices[df_prices['basic_unit_count'] == 1][['cohort_id', 'product_id', 'current_price']].drop_duplicates()
lookup = lookup.merge(base_prices, on=['product_id', 'cohort_id'], how='left')

# Merge stocks (sum across warehouses per product for display; warehouse-level for push filtering)
wh_df = pd.DataFrame(WAREHOUSE_MAPPING, columns=['region', 'warehouse', 'warehouse_id', 'cohort_id'])
product_stocks = df_stocks.groupby('product_id')['stocks'].sum().reset_index().rename(columns={'stocks': 'total_stocks'})
lookup = lookup.merge(product_stocks, on='product_id', how='left')
lookup['total_stocks'] = lookup['total_stocks'].fillna(0).astype(int)

print(f'Lookup table: {len(lookup)} rows (product x cohort)')
print(f'  With market data: {(lookup["price_tiers"].apply(len) > 0).sum()}')
print(f'  With WAC: {lookup["wac_p"].notna().sum()}')
print(f'  With target margin: {lookup["target_margin"].notna().sum()}')

Lookup table: 230639 rows (product x cohort)
  With market data: 42277
  With WAC: 75560
  With target margin: 230639


In [7]:
# =============================================================================
# DEFINE YOUR SKU ACTIONS HERE
# =============================================================================
# Format: (product_id, action)
#   action can be: 'market_min', 'market_25', 'market_50', 'market_75',
#                  'market_max', 'market_avg', 'target_margin',
#                  'step_up', 'step_down', or a number (fixed price)
#
# step_up: next tier above current price in the price_tiers list
# step_down: next tier below current price in the price_tiers list
# Each product is automatically expanded to ALL cohorts.
# Market-based actions use the cohort-specific market price.
# Only SKUs with stock > 0 are pushed.

PUSH_LIST = [
(1492,700,'step_down'),(6935,1124,'step_down'),(7630,1126,'step_down'),(88,704,'step_down'),(5493,703,'step_down'),(6935,701,'step_down'),(2778,1124,'step_down'),(6935,700,'step_down'),(6496,703,'step_down'),(201,701,'step_down'),(5493,1124,'step_down'),(5434,703,'step_down'),(11737,1123,'step_down'),(10993,701,'step_down'),(1492,704,'step_down'),(6935,1123,'step_down'),(5491,1123,'step_down'),(2537,704,'step_down'),(2497,1124,'step_down'),(2912,704,'step_down'),(616,703,'step_down'),(615,703,'step_down'),(1124,1124,'step_down'),(6497,1124,'step_down'),(1492,701,'step_down'),(624,704,'step_down'),(6496,1124,'step_down'),(25899,704,'step_down'),(10,704,'step_down'),(6494,1123,'step_down'),(5493,1123,'step_down'),(3961,700,'step_down'),(25899,702,'step_down'),(8286,1123,'step_down'),(6494,702,'step_down'),(624,703,'step_down'),(593,701,'step_down'),(6935,702,'step_down'),(10,700,'step_down'),(6494,704,'step_down'),(1492,1124,'step_down'),(5491,1126,'step_down'),(616,704,'step_down'),(1504,704,'step_down'),(1490,1125,'step_down'),(589,1123,'step_down'),(12924,704,'step_down'),(5491,703,'step_down'),(6935,704,'step_down'),(6936,700,'step_down'),(11396,703,'step_down'),(8283,1123,'step_down'),(6497,701,'step_down'),(1492,703,'step_down'),(6496,1125,'step_down'),(624,1123,'step_down'),(145,703,'step_down'),(12924,1125,'step_down'),(615,704,'step_down'),(12925,1124,'step_down'),(9208,703,'step_down'),(7886,1125,'step_down'),(2778,701,'step_down'),(6936,702,'step_down'),(6936,701,'step_down'),(5493,1126,'step_down'),(220,700,'step_down'),(8853,1123,'step_down'),(6936,1123,'step_down'),(6494,703,'step_down'),(3893,704,'step_down'),(8285,703,'step_down'),(11492,1123,'step_down'),(6494,701,'step_down'),(167,700,'step_down'),(990,703,'step_down'),(6494,1124,'step_down'),(12924,1124,'step_down'),(1523,703,'step_down'),(5081,701,'step_down'),(6497,1123,'step_down'),(11972,703,'step_down'),(11555,701,'step_down'),(593,703,'step_down'),(9827,700,'step_down'),(18,701,'step_down'),(3891,700,'step_down'),(9827,701,'step_down'),(24126,701,'step_down'),(10148,700,'step_down'),(990,1123,'step_down'),(215,703,'step_down'),(2744,701,'step_down'),(2537,700,'step_down'),(1082,701,'step_down'),(5435,700,'step_down'),(5769,704,'step_down'),(5490,703,'step_down'),(6935,703,'step_down'),(1490,702,'step_down'),(1523,704,'step_down'),(1429,702,'step_down'),(5152,1126,'step_down'),(82,1126,'step_down'),(12925,704,'step_down'),(4019,703,'step_down'),(25,704,'step_down'),(615,702,'step_down'),(616,702,'step_down'),(9070,1126,'step_down'),(616,1123,'step_down'),(141,1123,'step_down'),(411,700,'step_down'),(990,1125,'step_down'),(2879,704,'step_down'),(5005,701,'step_down'),(9208,704,'step_down'),(1492,1123,'step_down'),(12925,1123,'step_down'),(1124,704,'step_down'),(2438,703,'step_down'),(6494,1126,'step_down'),(471,703,'step_down'),(6936,1124,'step_down'),(6936,703,'step_down'),(4678,700,'step_down'),(8853,700,'step_down'),(24126,700,'step_down'),(4678,701,'step_down'),(11690,703,'step_down'),(12764,702,'step_down'),(12661,700,'step_down'),(26228,700,'step_down'),(3575,703,'step_down'),(7004,702,'step_down'),(3961,703,'step_down'),(1082,703,'step_down'),(8853,702,'step_down'),(4203,703,'step_down'),(215,1124,'step_down'),(5164,702,'step_down'),(1622,704,'step_down'),(1492,702,'step_down'),(2188,1123,'step_down'),(856,703,'step_down'),(12924,703,'step_down'),(9070,702,'step_down'),(10148,703,'step_down'),(205,1123,'step_down'),(10716,701,'step_down'),(2422,701,'step_down'),(361,1124,'step_down'),(4926,703,'step_down'),(84,1125,'step_down'),(3258,703,'step_down'),(11887,704,'step_down'),(5165,1124,'step_down'),(417,1126,'step_down'),(12923,1123,'step_down'),(12271,703,'step_down'),(823,1125,'step_down'),(6494,700,'step_down'),(12791,702,'step_down'),(25901,704,'step_down'),(220,1124,'step_down'),(9356,704,'step_down'),(589,1126,'step_down'),(25903,704,'step_down'),(5490,704,'step_down'),(12536,702,'step_down'),(3893,700,'step_down'),(6496,1123,'step_down'),(5111,1126,'step_down'),(12269,704,'step_down'),(12870,704,'step_down'),(9357,704,'step_down'),(6935,1126,'step_down'),(12925,703,'step_down'),(107,1124,'step_down'),(615,1126,'step_down'),(6496,1126,'step_down'),(143,1124,'step_down'),(83,703,'step_down'),(32,704,'step_down'),(141,1124,'step_down'),(3670,700,'step_down'),(3,703,'step_down'),(9530,704,'step_down'),(624,702,'step_down'),(201,703,'step_down'),(5490,702,'step_down'),(4927,703,'step_down'),(8853,701,'step_down'),(25899,1125,'step_down'),(2934,702,'step_down'),(624,1124,'step_down'),(2188,1126,'step_down'),(20927,703,'step_down'),(4019,704,'step_down'),(615,701,'step_down'),(1490,1124,'step_down'),(248,704,'step_down'),(13258,702,'step_down'),(3900,702,'step_down'),(7520,704,'step_down'),(965,700,'step_down'),(9070,703,'step_down'),(1124,702,'step_down'),(951,1123,'step_down'),(11689,1124,'step_down'),(345,703,'step_down'),(1339,701,'step_down'),(75,704,'step_down'),(2537,703,'step_down'),(8284,702,'step_down'),(17,704,'step_down'),(2916,1124,'step_down'),(1243,1125,'step_down'),(1490,703,'step_down'),(7036,704,'step_down'),(25900,704,'step_down'),(22089,1126,'step_down'),(590,702,'step_down'),(1446,704,'step_down'),(624,1125,'step_down'),(6497,1126,'step_down'),(412,1126,'step_down'),(20208,700,'step_down'),(5494,1123,'step_down'),(12128,702,'step_down'),(616,701,'step_down'),(2878,1124,'step_down'),(24127,701,'step_down'),(6931,1125,'step_down'),(11396,1125,'step_down'),(2436,700,'step_down'),(5494,703,'step_down'),(9373,1125,'step_down'),(6627,1123,'step_down'),(11427,1123,'step_down'),(4223,702,'step_down'),(3,1123,'step_down'),(12271,704,'step_down'),(12128,701,'step_down'),(5490,1124,'step_down'),(13490,703,'step_down'),(3,704,'step_down'),(8853,1125,'step_down'),(2436,703,'step_down'),(5152,703,'step_down'),(11894,704,'step_down'),(3994,703,'step_down'),(8285,1126,'step_down'),(952,704,'step_down'),(25904,703,'step_down'),(5723,701,'step_down'),(2805,704,'step_down'),(4225,701,'step_down'),(26519,702,'step_down'),(856,704,'step_down'),(624,1126,'step_down'),(11689,703,'step_down'),(417,704,'step_down'),(12764,1125,'step_down'),(10148,701,'step_down'),(336,704,'step_down'),(5084,701,'step_down'),(2935,702,'step_down'),(2878,703,'step_down'),(66,702,'step_down'),(12304,702,'step_down'),(201,704,'step_down'),(4224,1126,'step_down'),(440,703,'step_down'),(12270,702,'step_down'),(21454,704,'step_down'),(9070,1123,'step_down'),(3577,703,'step_down'),(3982,703,'step_down'),(12764,1123,'step_down'),(1490,704,'step_down'),(5966,700,'step_down'),(9529,1126,'step_down'),(10720,704,'step_down'),(216,1126,'step_down'),(6629,704,'step_down'),(1452,703,'step_down'),(146,1124,'step_down'),(107,702,'step_down'),(11685,1126,'step_down'),(20927,1124,'step_down'),(3898,701,'step_down'),(12662,700,'step_down'),(12128,1125,'step_down'),(20555,1126,'step_down'),(2912,703,'step_down'),(26228,1124,'step_down'),(26228,1123,'step_down'),(3982,704,'step_down'),(26228,703,'step_down'),(26228,1126,'step_down'),(26228,704,'step_down'),(26228,702,'step_down'),(26228,1125,'step_down'),(2538,704,'step_down'),(206,702,'step_down'),(7520,1124,'step_down'),(7520,701,'step_down'),(23,704,'step_down'),(1006,704,'step_down'),(957,703,'step_down'),(3979,704,'step_down'),(12304,1126,'step_down'),(9065,703,'step_down'),(12304,1124,'step_down'),(23992,1123,'step_down'),(10825,1126,'step_down'),(19687,1126,'step_down'),(9065,704,'step_down'),(3391,703,'step_down'),(589,1124,'step_down'),(8283,1124,'step_down'),(9209,703,'step_down'),(616,1126,'step_down'),(10825,1124,'step_down'),(3994,704,'step_down'),(417,702,'step_down'),(6158,703,'step_down'),(13045,703,'step_down'),(10124,1125,'step_down'),(10716,704,'step_down'),(11234,704,'step_down'),(20208,701,'step_down'),(951,1126,'step_down'),(1654,702,'step_down'),(6936,1126,'step_down'),(11425,702,'step_down'),(3980,704,'step_down'),(10720,703,'step_down'),(13115,703,'step_down'),(10719,703,'step_down'),(2369,701,'step_down'),(5494,1124,'step_down'),(10524,704,'step_down'),(3955,702,'step_down'),(12923,1126,'step_down'),(7520,1126,'step_down'),(9459,1123,'step_down'),(4958,700,'step_down'),(12925,701,'step_down'),(12677,701,'step_down'),(10719,704,'step_down'),(4715,701,'step_down'),(1492,1125,'step_down'),(6629,1126,'step_down'),(11885,702,'step_down'),(12508,700,'step_down'),(25342,701,'step_down'),(4223,1126,'step_down'),(615,700,'step_down'),(26104,1124,'step_down'),(382,703,'step_down'),(8853,703,'step_down'),(10717,700,'step_down'),(3673,1124,'step_down'),(23541,701,'step_down'),(14037,702,'step_down'),(4959,701,'step_down'),(12538,1126,'step_down'),(3891,703,'step_down'),(221,703,'step_down'),(1447,1124,'step_down'),(8120,700,'step_down'),(11738,703,'step_down'),(4956,701,'step_down'),(24844,700,'step_down'),(12270,704,'step_down'),(2934,1124,'step_down'),(201,702,'step_down'),(2803,700,'step_down'),(6936,1125,'step_down'),(1339,703,'step_down'),(2436,1124,'step_down'),(1446,1124,'step_down'),(12142,704,'step_down'),(2203,1123,'step_down'),(24724,702,'step_down'),(2049,700,'step_down'),(13114,701,'step_down'),(1082,704,'step_down'),(11091,702,'step_down'),(523,1124,'step_down'),(9070,1125,'step_down'),(11493,703,'step_down'),(11492,700,'step_down'),(6098,702,'step_down'),(3670,1124,'step_down'),(2497,700,'step_down'),(12661,704,'step_down'),(2424,702,'step_down'),(12539,1126,'step_down'),(9208,700,'step_down'),(2774,703,'step_down'),(5434,704,'step_down'),(8853,1126,'step_down'),(24126,703,'step_down'),(3597,703,'step_down'),(1084,701,'step_down'),(9783,703,'step_down'),(24127,700,'step_down'),(26226,700,'step_down'),(24124,701,'step_down'),(11454,1126,'step_down'),(3961,704,'step_down'),(12294,1126,'step_down'),(1243,702,'step_down'),(3575,1124,'step_down'),(24127,702,'step_down'),(12662,704,'step_down'),(11886,1123,'step_down'),(26519,701,'step_down'),(7705,703,'step_down'),(3016,1123,'step_down'),(214,702,'step_down'),(80,704,'step_down'),(12616,701,'step_down'),(11868,701,'step_down'),(4957,700,'step_down'),(13041,704,'step_down'),(12616,700,'step_down'),(12269,1125,'step_down'),(107,703,'step_down'),(10717,704,'step_down'),(2104,702,'step_down'),(944,700,'step_down'),(9065,1125,'step_down'),(965,1126,'step_down'),(12294,1125,'step_down'),(26225,701,'step_down'),(21068,1126,'step_down'),(3673,703,'step_down'),(9789,1126,'step_down'),(3016,703,'step_down'),(11793,703,'step_down'),(13117,703,'step_down'),(12128,1124,'step_down'),(11526,701,'step_down'),(8660,1123,'step_down'),(4883,701,'step_down'),(20498,704,'step_down'),(590,1123,'step_down'),(4678,704,'step_down'),(3575,704,'step_down'),(6630,1126,'step_down'),(4718,700,'step_down'),(412,1124,'step_down'),(1087,701,'step_down'),(1234,704,'step_down'),(24124,700,'step_down'),(3016,700,'step_down'),(19685,703,'step_down'),(2803,1123,'step_down'),(2262,1126,'step_down'),(5494,702,'step_down'),(4224,1123,'step_down'),(2497,1126,'step_down'),(3898,704,'step_down'),(26223,703,'step_down'),(26223,702,'step_down'),(26223,1125,'step_down'),(26223,704,'step_down'),(26223,1126,'step_down'),(26223,700,'step_down'),(26223,1124,'step_down'),(9786,1126,'step_down'),(72,703,'step_down'),(956,704,'step_down'),(2735,701,'step_down'),(22316,700,'step_down'),(12270,703,'step_down'),(13116,1126,'step_down'),(903,704,'step_down'),(22317,701,'step_down'),(26223,1123,'step_down'),(11059,701,'step_down'),(738,1126,'step_down'),(12923,703,'step_down'),(9207,704,'step_down'),(26227,700,'step_down'),(2879,1123,'step_down'),(5585,1124,'step_down'),(11059,704,'step_down'),(20208,704,'step_down'),(11133,1123,'step_down'),(1050,703,'step_down'),(11868,700,'step_down'),(2666,700,'step_down'),(5165,703,'step_down'),(19986,702,'step_down'),(33,701,'step_down'),(26223,701,'step_down'),(7705,1126,'step_down'),(9828,704,'step_down'),(17,701,'step_down'),(345,704,'step_down'),(2050,700,'step_down'),(616,700,'step_down'),(25,701,'step_down'),(12294,1124,'step_down'),(9085,1125,'step_down'),(5648,704,'step_down'),(1243,1123,'step_down'),(111,700,'step_down'),(11962,702,'step_down'),(12147,700,'step_down'),(45,704,'step_down'),(1243,1124,'step_down'),(2287,1124,'step_down'),(991,702,'step_down'),(8155,1125,'step_down'),(13490,702,'step_down'),(3673,1123,'step_down'),(11454,704,'step_down'),(91,700,'step_down'),(4001,703,'step_down'),(5966,1126,'step_down'),(13256,702,'step_down'),(11403,703,'step_down'),(3674,704,'step_down'),(8963,1123,'step_down'),(12128,700,'step_down'),(4965,700,'step_down'),(6936,704,'step_down'),(1451,1124,'step_down'),(23541,703,'step_down'),(11357,700,'step_down'),(2934,1126,'step_down'),(4955,701,'step_down'),(5651,703,'step_down'),(8907,701,'step_down'),(3,702,'step_down'),(24707,701,'step_down'),(12539,702,'step_down'),(6575,700,'step_down'),(3597,700,'step_down'),(26224,700,'step_down'),(107,704,'step_down'),(9945,700,'step_down'),(4225,1124,'step_down'),(18,702,'step_down'),(13116,703,'step_down'),(1448,704,'step_down'),(788,703,'step_down'),(11492,701,'step_down'),(4019,702,'step_down'),(2109,704,'step_down'),(7708,702,'step_down'),(13486,703,'step_down'),(6872,1126,'step_down'),(345,1126,'step_down'),(5452,701,'step_down'),(21088,1125,'step_down'),(415,703,'step_down'),(12543,700,'step_down'),(4001,704,'step_down'),(6629,1125,'step_down'),(2326,1123,'step_down'),(3462,703,'step_down'),(8106,700,'step_down'),(23331,701,'step_down'),(3462,702,'step_down'),(5493,700,'step_down'),(1450,1124,'step_down'),(11793,704,'step_down'),(3673,1126,'step_down'),(2108,700,'step_down'),(19985,701,'step_down'),(8157,1123,'step_down'),(23332,701,'step_down'),(24705,1126,'step_down'),(12147,1123,'step_down'),(23331,1126,'step_down'),(142,702,'step_down'),(12270,1123,'step_down'),(3955,1126,'step_down'),(10148,1126,'step_down'),(2726,703,'step_down'),(2417,701,'step_down'),(1055,1125,'step_down'),(12791,1126,'step_down'),(1655,1125,'step_down'),(791,703,'step_down'),(205,1124,'step_down'),(26229,1126,'step_down'),(5490,1125,'step_down'),(13115,701,'step_down'),(737,703,'step_down'),(10574,701,'step_down'),(5967,701,'step_down'),(20927,704,'step_down'),(6931,702,'step_down'),(965,701,'step_down'),(9830,703,'step_down'),(2104,704,'step_down'),(926,700,'step_down'),(26229,702,'step_down'),(25586,704,'step_down'),(418,1125,'step_down'),(9489,703,'step_down'),(23248,1124,'step_down'),(11972,702,'step_down'),(22634,703,'step_down'),(22665,704,'step_down'),(12553,700,'step_down'),(3892,703,'step_down'),(1453,703,'step_down'),(3898,702,'step_down'),(114,1126,'step_down'),(12609,701,'step_down'),(11738,702,'step_down'),(5209,702,'step_down'),(5152,702,'step_down'),(26229,703,'step_down'),(475,1125,'step_down'),(1598,703,'step_down'),(1492,1126,'step_down'),(25252,703,'step_down'),(24707,704,'step_down'),(13114,1125,'step_down'),(3597,1124,'step_down'),(3899,704,'step_down'),(4718,704,'step_down'),(11739,704,'step_down'),(475,703,'step_down'),(23331,702,'step_down'),(13114,700,'step_down'),(4225,703,'step_down'),(12270,1124,'step_down'),(10181,1125,'step_down'),(10574,702,'step_down'),(2916,1125,'step_down'),(26229,704,'step_down'),(26229,1125,'step_down'),(2271,702,'step_down'),(3597,1126,'step_down'),(411,1126,'step_down'),(1560,701,'step_down'),(11725,702,'step_down'),(1091,703,'step_down'),(214,704,'step_down'),(10574,1125,'step_down'),(1895,703,'step_down'),(26229,1124,'step_down'),(416,1126,'step_down'),(2803,1124,'step_down'),(20650,703,'step_down'),(107,1125,'step_down'),(9834,701,'step_down'),(11132,1123,'step_down'),(4717,701,'step_down'),(13023,1126,'step_down'),(3576,703,'step_down'),(12980,703,'step_down'),(6097,703,'step_down'),(6030,1125,'step_down'),(11403,1124,'step_down'),(11788,701,'step_down'),(2105,704,'step_down'),(5967,700,'step_down'),(3975,704,'step_down'),(4927,1123,'step_down'),(9374,1125,'step_down'),(965,703,'step_down'),(21091,1123,'step_down'),(440,704,'step_down'),(20650,701,'step_down'),(2287,703,'step_down'),(10491,701,'step_down'),(10276,701,'step_down'),(10531,703,'step_down'),(4962,700,'step_down'),(1050,1124,'step_down'),(3434,1124,'step_down'),(11969,700,'step_down'),(3893,703,'step_down'),(3049,703,'step_down'),(4963,701,'step_down'),(2049,704,'step_down'),(9827,704,'step_down'),(1339,1124,'step_down'),(4716,704,'step_down'),(94,1126,'step_down'),(790,1126,'step_down'),(996,701,'step_down'),(7520,1125,'step_down'),(12553,1126,'step_down'),(10994,704,'step_down'),(20650,1126,'step_down'),(1599,704,'step_down'),(3597,704,'step_down'),(4715,703,'step_down'),(641,700,'step_down'),(4019,1124,'step_down'),(414,703,'step_down'),(4715,704,'step_down'),(13023,701,'step_down'),(380,703,'step_down'),(4222,1126,'step_down'),(1134,702,'step_down'),(5152,1123,'step_down'),(5966,704,'step_down'),(10716,1123,'step_down'),(26229,1123,'step_down'),(4225,704,'step_down'),(143,1126,'step_down'),(13485,702,'step_down'),(4717,700,'step_down'),(9085,1124,'step_down'),(13256,704,'step_down'),(3597,702,'step_down'),(23994,1124,'step_down'),(21454,1125,'step_down'),(1576,1124,'step_down'),(22438,702,'step_down'),(19924,704,'step_down'),(11399,1123,'step_down'),(9609,703,'step_down'),(21454,1126,'step_down'),(24708,700,'step_down'),(12128,703,'step_down'),(11891,703,'step_down'),(3893,1123,'step_down'),(2803,703,'step_down'),(440,701,'step_down'),(2911,703,'step_down'),(1599,701,'step_down'),(3673,702,'step_down'),(2287,701,'step_down'),(217,1124,'step_down'),(35,701,'step_down'),(22973,1123,'step_down'),(3892,700,'step_down'),(11961,1125,'step_down'),(4225,1126,'step_down'),(307,704,'step_down'),(13114,1126,'step_down'),(217,1125,'step_down'),(4225,1125,'step_down'),(9789,704,'step_down'),(4043,701,'step_down'),(4225,702,'step_down'),(23491,701,'step_down'),(11971,1124,'step_down'),(150,1124,'step_down'),(24709,700,'step_down'),(6627,701,'step_down'),(13402,701,'step_down'),(225,702,'step_down'),(8944,1126,'step_down'),(9356,702,'step_down'),(10983,1124,'step_down'),(9613,702,'step_down'),(20961,701,'step_down'),(11971,702,'step_down'),(11971,703,'step_down'),(5153,1125,'step_down'),(5153,1124,'step_down'),(5577,703,'step_down'),(13117,701,'step_down'),(1598,704,'step_down'),(3901,703,'step_down'),(22667,1123,'step_down'),(10276,700,'step_down'),(9789,701,'step_down'),(9789,700,'step_down'),(5585,700,'step_down'),(4867,1123,'step_down'),(9789,702,'step_down')

]


# =============================================================================
# COMPUTE NEW PRICES (per product × cohort)
# =============================================================================
ACTION_TO_COLUMN = {
    'market_min': 'market_min',
    'market_25': 'market_25',
    'market_50': 'market_50',
    'market_75': 'market_75',
    'market_max': 'market_max',
    'market_avg': 'market_avg',
}

results = []
for product_id,cohort_id, action in PUSH_LIST:
    product_rows = lookup[(lookup['product_id'] == product_id)&(lookup['cohort_id'] == cohort_id)]
    if product_rows.empty:
        results.append({'product_id': product_id, 'cohort_id': '-', 'action': str(action), 'status': 'NOT FOUND'})
        continue

    for _, row in product_rows.iterrows():
        cohort_id = int(row['cohort_id'])
        wac = row.get('wac_p', None)
        sku = row.get('sku', '')
        brand = row.get('brand', '')

        if isinstance(action, (int, float)):
            base_price = float(action)
            action_label = f'fixed_{action}'
        elif action == 'target_margin':
            tm = row.get('target_margin', 0.10)
            if pd.isna(wac) or wac <= 0:
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'NO WAC'})
                continue
            base_price = wac / (1 - tm)
            action_label = f'target_margin ({tm:.1%})'
        elif action == 'step_up':
            tiers = row.get('price_tiers', [])
            cur = row.get('current_price', 0)
            if not tiers or pd.isna(cur) or cur <= 0:
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'NO TIERS OR PRICE'})
                continue
            next_up = [t for t in tiers if t > cur + 0.25]
            if not next_up:
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'ALREADY AT TOP'})
                continue
            base_price = next_up[0]
            action_label = f'step_up ({cur:.2f} -> {base_price:.2f})'
        elif action == 'step_down':
            tiers = row.get('price_tiers', [])
            cur = row.get('current_price', 0)
            if not tiers or pd.isna(cur) or cur <= 0:
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'NO TIERS OR PRICE'})
                continue
            next_down = [t for t in reversed(tiers) if t < cur - 0.25]
            if not next_down:
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'ALREADY AT BOTTOM'})
                continue
            base_price = next_down[0]
            action_label = f'step_down ({cur:.2f} -> {base_price:.2f})'
        elif action in ACTION_TO_COLUMN:
            col = ACTION_TO_COLUMN[action]
            val = row.get(col, None)
            if pd.isna(val):
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'NO MARKET DATA'})
                continue
            base_price = val
            action_label = action
        else:
            results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': str(action), 'status': 'INVALID ACTION'})
            continue

        base_price_rounded = np.round(base_price * 4) / 4
        margin = (base_price_rounded - wac) / base_price_rounded if (wac and wac > 0 and base_price_rounded > 0) else None

        cur_price = row.get('current_price', None)

        results.append({
            'product_id': product_id,
            'cohort_id': cohort_id,
            'sku': sku,
            'brand': brand,
            'action': action_label,
            'wac': wac,
            'current_price': cur_price,
            'base_price': base_price,
            'new_price': base_price_rounded,
            'margin': margin,
            'status': 'OK',
        })

df_results = pd.DataFrame(results)

ok_count = (df_results['status'] == 'OK').sum()
err_count = len(df_results) - ok_count
products_ok = df_results[df_results['status'] == 'OK']['product_id'].nunique()
print(f'Results: {ok_count} OK across {products_ok} products × cohorts, {err_count} errors/skipped')
if err_count > 0:
    print('\nErrors/Skipped:')
    print(df_results[df_results['status'] != 'OK'][['product_id', 'cohort_id', 'sku', 'action', 'status']].to_string(index=False))
df_results[df_results['status'] == 'OK']

Results: 260 OK across 182 products × cohorts, 518 errors/skipped

Errors/Skipped:
 product_id  cohort_id                                                                                     sku    action            status
       1492        700                                                                  زيت حبوبة خليط - 1 لتر step_down ALREADY AT BOTTOM
       5493        703                                                          شويبس جولد اناناس جيب - 240 مل step_down ALREADY AT BOTTOM
       2778       1124                                                                    كوكا كولا - 2.45 لتر step_down ALREADY AT BOTTOM
        201        701                                                            عصير تانج برتقال ظرف - 20 جم step_down ALREADY AT BOTTOM
       5493       1124                                                          شويبس جولد اناناس جيب - 240 مل step_down ALREADY AT BOTTOM
       5434        703                                                             

,product_id,cohort_id,sku,action,status,brand,wac,current_price,base_price,new_price,margin
1,6935,1124,كوكاكولا اكشن - 300 مل,step_down (120.00 -> 118.00),OK,كوكا كولا,116.847788,120.00,118.00,118.00,0.009765
2,7630,1126,فيورى جولد - 400 مل,step_down (243.50 -> 240.00),OK,فيوري,174.693518,243.50,240.00,240.00,0.272110
3,88,704,شاى ليبتون ناعم - 250 جم,step_down (51.50 -> 50.75),OK,ليبتون,48.621300,51.50,50.75,50.75,0.041945
5,6935,701,كوكاكولا اكشن - 300 مل,step_down (120.00 -> 118.00),OK,كوكا كولا,116.847788,120.00,118.00,118.00,0.009765
7,6935,700,كوكاكولا اكشن - 300 مل,step_down (120.00 -> 118.00),OK,كوكا كولا,116.847788,120.00,118.00,118.00,0.009765
...,...,...,...,...,...,...,...,...,...,...,...
758,8944,1126,طحينة البوادى ظرف - 42 جم,step_down (131.75 -> 130.00),OK,البوادي,127.108800,131.75,130.00,130.00,0.022240
762,20961,701,ايبون بنكهه الفاكهه بكريمه الحليب - 4 كجم,step_down (650.00 -> 614.75),OK,أيبون,545.412201,650.00,614.75,614.75,0.112790
765,5153,1125,ريد بل ابيض بجوز الهند و التوت - 250 مل,step_down (1220.00 -> 1210.00),OK,ريد بل,1170.550561,1220.00,1210.00,1210.00,0.032603
769,1598,704,بطارية قلم افيريدى اسود - 20 حجر,step_down (241.50 -> 238.25),OK,افيريدي,218.286468,241.50,238.25,238.25,0.083792


In [8]:
df_results=df_results[df_results['current_price']>df_results['new_price']]
df_results

,product_id,cohort_id,sku,action,status,brand,wac,current_price,base_price,new_price,margin
1,6935,1124,كوكاكولا اكشن - 300 مل,step_down (120.00 -> 118.00),OK,كوكا كولا,116.847788,120.00,118.00,118.00,0.009765
2,7630,1126,فيورى جولد - 400 مل,step_down (243.50 -> 240.00),OK,فيوري,174.693518,243.50,240.00,240.00,0.272110
3,88,704,شاى ليبتون ناعم - 250 جم,step_down (51.50 -> 50.75),OK,ليبتون,48.621300,51.50,50.75,50.75,0.041945
5,6935,701,كوكاكولا اكشن - 300 مل,step_down (120.00 -> 118.00),OK,كوكا كولا,116.847788,120.00,118.00,118.00,0.009765
7,6935,700,كوكاكولا اكشن - 300 مل,step_down (120.00 -> 118.00),OK,كوكا كولا,116.847788,120.00,118.00,118.00,0.009765
...,...,...,...,...,...,...,...,...,...,...,...
758,8944,1126,طحينة البوادى ظرف - 42 جم,step_down (131.75 -> 130.00),OK,البوادي,127.108800,131.75,130.00,130.00,0.022240
762,20961,701,ايبون بنكهه الفاكهه بكريمه الحليب - 4 كجم,step_down (650.00 -> 614.75),OK,أيبون,545.412201,650.00,614.75,614.75,0.112790
765,5153,1125,ريد بل ابيض بجوز الهند و التوت - 250 مل,step_down (1220.00 -> 1210.00),OK,ريد بل,1170.550561,1220.00,1210.00,1210.00,0.032603
769,1598,704,بطارية قلم افيريدى اسود - 20 حجر,step_down (241.50 -> 238.25),OK,افيريدي,218.286468,241.50,238.25,238.25,0.083792


In [9]:
# =============================================================================
# PUSH PRICES
# =============================================================================
MODE = 'live'  # Change to 'live' to actually push

df_ok = df_results[df_results['status'] == 'OK'].copy()

# Filter to only SKUs with stock
if len(df_ok) > 0:
    stock_by_product = df_stocks.groupby('product_id')['stocks'].sum().reset_index()
    df_ok = df_ok.merge(stock_by_product, on='product_id', how='left')
    no_stock = df_ok['stocks'].fillna(0) <= 0
    if no_stock.any():
        print(f'Skipping {no_stock.sum()} rows with no stock')
        df_ok = df_ok[~no_stock].copy()

if len(df_ok) == 0:
    print('No valid prices to push.')
else:
    wh_df = pd.DataFrame(WAREHOUSE_MAPPING, columns=['region', 'warehouse', 'warehouse_id', 'cohort_id'])

    push_rows = []
    for _, r in df_ok.iterrows():
        target_cohort = int(r['cohort_id'])
        cohort_whs = wh_df[wh_df['cohort_id'] == target_cohort]

        if cohort_whs.empty:
            cohort_whs = pd.DataFrame([{'warehouse_id': 1, 'cohort_id': target_cohort}])

        cur_price = r['current_price'] if pd.notna(r.get('current_price')) else 0

        for _, wh in cohort_whs.iterrows():
            push_rows.append({
                'product_id': int(r['product_id']),
                'sku': r['sku'],
                'new_price': r['new_price'],
                'warehouse_id': int(wh['warehouse_id']),
                'cohort_id': target_cohort,
                'stocks': 1,
                'current_price': cur_price,
            })

    df_push = pd.DataFrame(push_rows)

    n_products = df_ok['product_id'].nunique()
    n_cohorts = df_ok['cohort_id'].nunique()
    print(f'Pushing {n_products} products × {n_cohorts} cohorts = {len(df_push)} rows')
    print(f'Mode: {MODE}')
    print()

    result = push_prices(
        df_prices=df_push,
        pus=df_pus,
        source_module='manual_push',
        mode=MODE,
    )
    print(f'\nDone. Result: {result}')

Pushing 182 products × 9 cohorts = 401 rows
Mode: live


🚀 MODE: LIVE
   Files will be prepared AND uploaded to API
Loading disable_pu_visibility from Google Sheets...
  ✓ Loaded 88 products to disable min PU visibility

PUSH PRICES - Source: manual_push
Total received: 401
Price changes to push: 401
Skipped (no change): 0

Price changes summary:
  Increases: 0
  Decreases: 401

🔗 Mirrored prices to 6 main/general cohorts (+259 rows)
   Cohort 695 ← 700: 45 rows
   Cohort 61 ← 700: 45 rows
   Cohort 699 ← 702: 27 rows
   Cohort 697 ← 703: 63 rows
   Cohort 698 ← 704: 61 rows
   Cohort 696 ← 1123: 18 rows

📋 Prepared 602 packing unit prices (incl. main cohorts)

Processing cohort: 61
  Saved: uploads/manual_push_61.xlsx (45 rows)
  Split into 1 chunks (size: 2000)


  Saving chunks: 100%|██████████| 1/1 [00:00<00:00, 139.73it/s]

  Uploading...


    ✓ Chunk 1 uploaded successfully

Processing cohort: 695
  Saved: uploads/manual_push_695.xlsx (45 rows)
  Split into 1 chunks (size: 4000)


  Saving chunks: 100%|██████████| 1/1 [00:00<00:00, 118.88it/s]

...

  Uploading...


    ✓ Chunk 1 uploaded successfully

Processing cohort: 1126
  Saved: uploads/manual_push_1126.xlsx (18 rows)
  Split into 1 chunks (size: 4000)


  Saving chunks: 100%|██████████| 1/1 [00:00<00:00, 194.96it/s]

  Uploading...


    ✓ Chunk 1 uploaded successfully

🚀 UPLOAD COMPLETE
Mode: live
Total prepared: 602
Total failed: 0
  Cleanup: removed 32 temporary .xlsx files from 2 folder(s)

Done. Result: {'total_received': 401, 'price_changes': 401, 'pushed': 602, 'failed': 0, 'skipped': 0, 'source_module': 'manual_push', 'timestamp': '2026-04-08 11:33:34', 'mode': 'live', 'skipped_cohorts': []}
